In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from scipy import stats
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# ───────────────────────── CFG ─────────────────────────

CFG = dict(
    CMD_DYN_THR  = {19: 0.5, 20: 1.5, 21: 2.0, 22: 1.0},
    CMD_GAP_THR  = {19: 30,  20: 10,  21: 30,  22: 30},
    N_FOLDS      = 5,
    EARLY_STOP   = 70,
    MAX_ROUNDS   = 15000,
    SEED         = 24,
    TARGET       = "bk_mean",
    ROW_ID_COL   = "row_id",
    WELCH_ALPHA  = 0.05,
    TRAIN_PATH   = "/kaggle/input/competitions/aiclubdatathon-26/train.csv",
    TEST_PATH    = "/kaggle/input/competitions/aiclubdatathon-26/test.csv",
)

LGB1 = dict(
    objective="regression_l1", metric="mae", verbose=-1,
    seed=CFG["SEED"], device="gpu", feature_pre_filter=False,
    learning_rate=0.03, num_leaves=63, min_child_samples=20,
    feature_fraction=0.5, bagging_fraction=0.8, bagging_freq=1,
    lambda_l1=0.5, lambda_l2=0.5, max_depth=8,
)
LGB2 = dict(
    objective="regression_l1", metric="mae", verbose=-1,
    seed=CFG["SEED"]+1, device="gpu", feature_pre_filter=False,
    learning_rate=0.025, num_leaves=255, min_child_samples=15,
    feature_fraction=0.8, bagging_fraction=0.7, bagging_freq=2,
    lambda_l1=0.1, lambda_l2=1.0, max_depth=12,
)
XGB1 = dict(
    objective="reg:absoluteerror", eval_metric="mae", verbosity=0,
    device="cuda", tree_method="hist", seed=CFG["SEED"],
    learning_rate=0.03, max_depth=6, min_child_weight=15,
    subsample=0.8, colsample_bytree=0.6, colsample_bylevel=0.8,
    reg_alpha=0.5, reg_lambda=1.0, gamma=0.1,
)
XGB2 = dict(
    objective="reg:absoluteerror", eval_metric="mae", verbosity=0,
    device="cuda", tree_method="hist", seed=CFG["SEED"]+1,
    learning_rate=0.025, max_depth=9, min_child_weight=5,
    subsample=0.7, colsample_bytree=0.9, colsample_bylevel=0.6,
    reg_alpha=0.1, reg_lambda=0.5, gamma=0.5,
)
CAT1 = dict(
    loss_function="MAE", eval_metric="MAE", task_type="GPU",
    random_seed=CFG["SEED"], verbose=0,
    early_stopping_rounds=CFG["EARLY_STOP"], iterations=CFG["MAX_ROUNDS"],
    learning_rate=0.03, depth=6, l2_leaf_reg=5,
    min_data_in_leaf=20, bagging_temperature=0.5,
    random_strength=1.0, border_count=128,
)
CAT2 = dict(
    loss_function="MAE", eval_metric="MAE", task_type="GPU",
    random_seed=CFG["SEED"]+1, verbose=0,
    early_stopping_rounds=CFG["EARLY_STOP"], iterations=CFG["MAX_ROUNDS"],
    learning_rate=0.02, depth=8, l2_leaf_reg=1,
    min_data_in_leaf=10, bagging_temperature=1.0,
    random_strength=2.0, border_count=255,
)
LR1_ALPHA = 1.0
LR2_ALPHA = 10.0

# ───────────────────── Feature listeleri ─────────────────────

BLK_FEATURES = [
    # Kimlik / kategorik
    "machineid", "commandno", "cmd_machine",
    "prgno", "fabric_weight", "dosage_curve", "cmd_rep",
    # Dinamizm
    "is_dynamic", "was_dynamic", "dyn_transition",
    # Sıra / ilerleme
    "blk_rank", "blk_rank_rev",
    "log_cum_dur", "proc_total_dur",
    "proc_n_blocks", "proc_n_dynamic",
    # Blok boyutu
    "n_rows", "block_dur_s", "dur_ratio",
    # Proses seviyesi
    "proc_bk_range", "proc_bk_start", "proc_tgt_mean",
    # Gap
    "gap_before_s", "log_gap", "log_gap1", "log_gap3",
    "gap_x_bk_lag1", "gap_x_tgt", "gap_roll3_mean",
    "log_total_gap", "cmd_gap_ratio", "gap_x_dynamic", "gap_x_delta",
    # Lag değerleri
    "bk_mean_lag1", "bk_mean_lag2", "bk_mean_lag3", "bk_mean_lag4", "bk_mean_lag5",
    "bk_last_lag1", "bk_last_lag2",
    "bk_std_lag1",
    "gap_lag1", "gap_lag3",
    "dur_lag1",
    # Momentum / rolling / range
    "bk_delta1", "bk_delta2", "bk_delta3", "bk_accel", "bk_jerk",
    "bk_roll3_mean", "bk_roll5_mean", "bk_roll5_std",
    "bk_range",
    "bk_filling", "bk_draining",
    "bk_momentum_dev",
    "rank_x_delta",
    "fill_rate",
    # Hedef takip
    "bk_tgt_mean", "bk_tgt_last", "bk_tgt_first",
    "bk_tgt_lag1", "bk_tgt_lag2",
    "tgt_delta1", "bk_tgt_error", "bk_to_tgt_ratio",
    "bk_tgt_gap_pct",
    "tgt_delta2", "tgt_accel",
    "tgt_roll3_mean", "tgt_roll3_std",
    "tgt_error_delta", "tgt_error_roll3",
    "eta_to_target",
    # Global istatistikler
    "cmd_bk_mean_global", "cmd_bk_std_global", "cmd_gap_mean_global",
    "cmd_bk_med_global",
    "cmd_tgt_error_mean", "cmd_tgt_error_std",
    # Sensör
    "kk_mean", "kk_last", "ak_mean",
    # Proses geçmişi
    "prev_proc_bk_mean", "prev_proc_bk_last",
    "log_hist_rows",
    # Vana oranları
    "bk_irtibat_ratio",
    "fast_dosage_ratio",
    "slow_dosage_ratio",
    "bk_valve_ratio",
    # Robot sinyalleri
    "kk_robot_mean",
    "bk_robot_mean",
    # KK target ve vana agg’leri
    "kk_tgt_mean",
    "kk_dosage_v", "kk_irtibat_v", "kk_bk_common",
]

BLK_CAT = ["machineid", "commandno", "cmd_machine",
           "prgno", "dosage_curve", "cmd_rep"]

VALVE_COLS = [
    "bk_irtibat_valve",
    "fast_dosage_valve",
    "slow_dosage_valve",
    "bk_dosage_valve",
    "kk_dosage_valve",
    "kk_irtibat_valve",
    "kk_bk_common_discharge",
]

# ───────────────────── Helper fonksiyonlar ─────────────────────

def cast_float32(df: pd.DataFrame) -> pd.DataFrame:
    for c in df.select_dtypes(include="float64").columns:
        df[c] = df[c].astype(np.float32)
    return df

def add_proc_key(df: pd.DataFrame) -> pd.DataFrame:
    df["proc_key"] = (
        df["machineid"].astype(str) + "_" +
        df["batchkey"].astype(str) + "_" +
        df["commandno"].astype(str)
    )
    return df

def fill_missing_sensor_cols(df: pd.DataFrame) -> pd.DataFrame:
    for c in ["bk_level", "bk_target_level", "kk_level", "ak_level",
              "kk_mikser_robotu", "bk_mikser_robotu"]:
        if c not in df.columns:
            df[c] = 0.0
    return df

def weighted_mae(y_true, y_pred, weights) -> float:
    w = np.asarray(weights, dtype=np.float64)
    return float(np.sum(w * np.abs(np.asarray(y_true) - np.asarray(y_pred))) / w.sum())

# ───────────────────── Vana oranları ─────────────────────

def compute_valve_ratios(df_block: pd.DataFrame) -> pd.Series:
    df_block = df_block.sort_values("timestamp")
    df_block["delta_bk"] = df_block["bk_level"].diff().fillna(0.0)

    eps = 1e-3
    total_delta = df_block["delta_bk"].sum()

    def on(col):
        if col not in df_block.columns:
            return pd.Series(False, index=df_block.index)
        return df_block[col] > 0

    irt_on_bk   = on("bk_irtibat_valve")
    fast_on_bk  = on("fast_dosage_valve")
    slow_on_bk  = on("slow_dosage_valve")
    any_on_bk   = irt_on_bk | fast_on_bk | slow_on_bk

    bk_irtibat_ratio  = df_block.loc[irt_on_bk,  "delta_bk"].sum() / (total_delta + eps)
    fast_dosage_ratio = df_block.loc[fast_on_bk, "delta_bk"].sum() / (total_delta + eps)
    slow_dosage_ratio = df_block.loc[slow_on_bk, "delta_bk"].sum() / (total_delta + eps)
    bk_valve_ratio    = df_block.loc[any_on_bk,  "delta_bk"].sum() / (total_delta + eps)

    return pd.Series(dict(
        bk_irtibat_ratio  = float(bk_irtibat_ratio),
        fast_dosage_ratio = float(fast_dosage_ratio),
        slow_dosage_ratio = float(slow_dosage_ratio),
        bk_valve_ratio    = float(bk_valve_ratio),
    ))

# ───────────────────── Block tespiti & aggregation ─────────────────────

def detect_and_aggregate_blocks(df_sorted: pd.DataFrame) -> pd.DataFrame:
    df = df_sorted.copy()
    df = fill_missing_sensor_cols(df)

    df["dt"] = (
        df.groupby("proc_key")["timestamp"]
          .diff().dt.total_seconds().fillna(0)
    )

    def rolling_std_by_cmd(grp):
        thr = CFG["CMD_DYN_THR"].get(int(grp["commandno"].iloc[0]), 1.0)
        return (grp["bk_level"].rolling(5, min_periods=2)
                .std().fillna(0) > thr).astype(int)

    df["is_dynamic"] = df.groupby("proc_key", group_keys=False).apply(rolling_std_by_cmd)

    df["time_break"] = df.apply(
        lambda r: int(r["dt"] > CFG["CMD_GAP_THR"].get(int(r["commandno"]), 30)),
        axis=1
    )
    first_idx = df.groupby("proc_key").head(1).index
    df.loc[first_idx, "time_break"] = 1

    df["region_change"] = (
        df.groupby("proc_key")["is_dynamic"]
          .diff().fillna(0).abs().astype(int)
    )
    df["new_block"] = ((df["time_break"] == 1) |
                       (df["region_change"] == 1)).astype(int)
    df.loc[first_idx, "new_block"] = 1

    df["block_id"]  = df.groupby("proc_key")["new_block"].cumsum()
    df["block_key"] = df["proc_key"] + "_b" + df["block_id"].astype(str)

    agg_dict = dict(
        proc_key     = ("proc_key",   "first"),
        machineid    = ("machineid",  "first"),
        commandno    = ("commandno",  "first"),
        block_id     = ("block_id",   "first"),
        t_start      = ("timestamp",  "first"),
        t_end        = ("timestamp",  "last"),
        n_rows       = ("bk_level",   "count"),
        bk_mean      = ("bk_level",   "mean"),
        bk_std       = ("bk_level",   "std"),
        bk_first     = ("bk_level",   "first"),
        bk_last      = ("bk_level",   "last"),
        bk_min       = ("bk_level",   "min"),
        bk_max       = ("bk_level",   "max"),
        bk_tgt_mean  = ("bk_target_level", "mean"),
        bk_tgt_last  = ("bk_target_level", "last"),
        bk_tgt_first = ("bk_target_level", "first"),
        is_dynamic   = ("is_dynamic", "max"),
        gap_before_s = ("dt",         "first"),
        kk_mean      = ("kk_level",   "mean"),
        kk_last      = ("kk_level",   "last"),
        ak_mean      = ("ak_level",   "mean"),
        kk_robot_mean = ("kk_mikser_robotu", "mean"),
        bk_robot_mean = ("bk_mikser_robotu", "mean"),
    )

    extra_cols = [
        ("fabric_weight",          "fabric_weight", "first"),
        ("prgno",                  "prgno",         "first"),
        ("kk_target_level",        "kk_tgt_mean",   "mean"),
        ("kk_dosage_valve",        "kk_dosage_v",   "mean"),
        ("kk_irtibat_valve",       "kk_irtibat_v",  "mean"),
        ("kk_bk_common_discharge", "kk_bk_common",  "mean"),
        ("dosage_curve_type",      "dosage_curve",  "first"),
        ("command_repetition",     "cmd_rep",       "first"),
    ]
    for src, dst, how in extra_cols:
        if src in df.columns:
            agg_dict[dst] = (src, how)

    agg = df.groupby("block_key").agg(**agg_dict).reset_index()

    agg["bk_std"]      = agg["bk_std"].fillna(0)
    agg["block_dur_s"] = (agg["t_end"] - agg["t_start"]).dt.total_seconds()

    agg = agg.sort_values(["proc_key", "block_id"]).reset_index(drop=True)
    agg["gap_before_s"] = (
        agg.groupby("proc_key")["t_start"]
           .transform(lambda x: x.diff().dt.total_seconds()).fillna(0)
    )

    for c in ["kk_mean", "kk_last", "ak_mean",
              "kk_robot_mean", "bk_robot_mean"]:
        if c not in agg.columns:
            agg[c] = 0.0

    valve_stats = (
        df.groupby("block_key", group_keys=True)
          .apply(compute_valve_ratios)
          .reset_index()
    )
    agg = agg.merge(
        valve_stats[["block_key",
                     "bk_irtibat_ratio",
                     "fast_dosage_ratio",
                     "slow_dosage_ratio",
                     "bk_valve_ratio"]],
        on="block_key", how="left"
    )

    return agg

# ───────────────────── Proses geçmişi & CMD istatistik ─────────────────────

def build_proc_prior(blk_df: pd.DataFrame) -> pd.DataFrame:
    ps = (blk_df.groupby("proc_key")
          .agg(machineid=("machineid", "first"),
               commandno=("commandno", "first"),
               t_start=("t_start", "first"),
               bk_mean_p=("bk_mean", "mean"),
               bk_last_p=("bk_last", "last"))
          .reset_index()
          .sort_values(["machineid", "commandno", "t_start"]))
    ps["prev_proc_bk_mean"] = (
        ps.groupby(["machineid", "commandno"])["bk_mean_p"].shift(1)
    )
    ps["prev_proc_bk_last"] = (
        ps.groupby(["machineid", "commandno"])["bk_last_p"].shift(1)
    )
    return ps[["proc_key", "prev_proc_bk_mean", "prev_proc_bk_last"]].fillna(0)

def build_cmd_stats(blk_df: pd.DataFrame) -> pd.DataFrame:
    base = (blk_df.groupby(["machineid", "commandno"])
            .agg(cmd_bk_mean_global=("bk_mean", "mean"),
                 cmd_bk_std_global=("bk_mean", "std"),
                 cmd_gap_mean_global=("gap_before_s", "mean"))
            .reset_index())
    med = (blk_df.groupby(["machineid", "commandno"])["bk_mean"]
           .median().reset_index(name="cmd_bk_med_global"))

    blk_df2 = blk_df.copy()
    blk_df2["_tgt_err"] = blk_df2["bk_tgt_mean"] - blk_df2["bk_mean"]
    err = (blk_df2.groupby(["machineid", "commandno"])["_tgt_err"]
           .agg(cmd_tgt_error_mean="mean", cmd_tgt_error_std="std")
           .reset_index())

    return base.merge(med, on=["machineid", "commandno"]).merge(err, on=["machineid", "commandno"])

# ───────────────────── Block-level feature engineering ─────────────────────

def build_features(blk_df: pd.DataFrame,
                   cmd_stats_df: pd.DataFrame = None,
                   proc_prior_df: pd.DataFrame = None) -> pd.DataFrame:
    b = blk_df.copy().sort_values(["proc_key", "block_id"]).reset_index(drop=True)

    # Proses meta
    proc_meta = (b.groupby("proc_key")
                 .agg(proc_n_blocks=("block_id", "count"),
                      proc_n_dynamic=("is_dynamic", "sum"),
                      proc_bk_range=("bk_mean", lambda x: x.max() - x.min()),
                      proc_bk_start=("bk_mean", "first"),
                      proc_tgt_mean=("bk_tgt_mean", "mean"),
                      proc_total_gap=("gap_before_s", "sum"),
                      proc_total_dur=("block_dur_s", "sum"))
                 .reset_index())
    b = b.merge(proc_meta, on="proc_key", how="left")

    # Komut istatistikleri
    if cmd_stats_df is not None:
        b = b.merge(cmd_stats_df, on=["machineid", "commandno"], how="left")
    else:
        for c in ["cmd_bk_mean_global", "cmd_bk_std_global", "cmd_gap_mean_global",
                  "cmd_bk_med_global", "cmd_tgt_error_mean", "cmd_tgt_error_std"]:
            b[c] = 0.0

    # Proses geçmişi
    if proc_prior_df is not None:
        b = b.merge(proc_prior_df, on="proc_key", how="left")
    else:
        b["prev_proc_bk_mean"] = 0.0
        b["prev_proc_bk_last"] = 0.0
    b[["prev_proc_bk_mean", "prev_proc_bk_last"]] = (
        b[["prev_proc_bk_mean", "prev_proc_bk_last"]].fillna(0)
    )

    grp = b.groupby("proc_key")

    # Lag’ler
    for lag in range(1, 6):
        b[f"bk_mean_lag{lag}"] = grp["bk_mean"].shift(lag)
        b[f"bk_last_lag{lag}"] = grp["bk_last"].shift(lag)
        b[f"bk_std_lag{lag}"]  = grp["bk_std"].shift(lag)
        b[f"bk_tgt_lag{lag}"]  = grp["bk_tgt_mean"].shift(lag)
        b[f"gap_lag{lag}"]     = grp["gap_before_s"].shift(lag)
        b[f"dur_lag{lag}"]     = grp["block_dur_s"].shift(lag)
        b[f"dyn_lag{lag}"]     = grp["is_dynamic"].shift(lag)
        b[f"nrows_lag{lag}"]   = grp["n_rows"].shift(lag)

    # Momentum
    b["bk_delta1"] = b["bk_mean"] - b["bk_mean_lag1"]
    b["bk_delta2"] = b["bk_mean_lag1"] - b["bk_mean_lag2"]
    b["bk_delta3"] = b["bk_mean_lag2"] - b["bk_mean_lag3"]
    b["bk_accel"]  = b["bk_delta1"] - b["bk_delta2"]
    b["bk_jerk"]   = b["bk_accel"] - (b["bk_delta2"] - b["bk_delta3"])

    # Rolling
    b["bk_roll3_mean"]  = grp["bk_mean"].transform(
        lambda x: x.shift(1).rolling(3, min_periods=1).mean()
    )
    b["bk_roll5_mean"]  = grp["bk_mean"].transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )
    b["bk_roll5_std"]   = grp["bk_mean"].transform(
        lambda x: x.shift(1).rolling(5, min_periods=2).std().fillna(0)
    )
    b["gap_roll3_mean"] = grp["gap_before_s"].transform(
        lambda x: x.shift(1).rolling(3, min_periods=1).mean()
    )

    lag1  = b["bk_mean_lag1"].fillna(0)
    roll5 = b["bk_roll5_mean"].fillna(lag1)

    # Momentum pattern / yön
    b["bk_momentum_dev"] = lag1 - roll5
    b["bk_filling"]      = (b["bk_delta1"].fillna(0) > 1).astype(int)
    b["bk_draining"]     = (b["bk_delta1"].fillna(0) < -1).astype(int)

    # Gap türevleri
    b["log_gap"]       = np.log1p(b["gap_before_s"])
    b["log_gap1"]      = np.log1p(b["gap_lag1"].fillna(0))
    b["log_gap3"]      = np.log1p(b["gap_lag3"].fillna(0))
    b["log_total_gap"] = np.log1p(b["proc_total_gap"])
    b["gap_x_bk_lag1"] = b["gap_before_s"] * lag1 / 3600
    b["gap_x_tgt"]     = b["gap_before_s"] * b["bk_tgt_mean"] / 3600
    b["cmd_gap_ratio"] = b["gap_before_s"] / (b["cmd_gap_mean_global"].fillna(1) + 1)
    b["gap_x_dynamic"] = b["gap_before_s"] * b["is_dynamic"]
    b["gap_x_delta"]   = b["gap_before_s"] * b["bk_delta1"].fillna(0).abs()

    # Hedef takip
    b["bk_tgt_error"] = b["bk_tgt_mean"] - lag1
    b["tgt_delta1"]   = b["bk_tgt_mean"] - b["bk_tgt_lag1"].fillna(0)
    b["bk_to_tgt_ratio"] = np.where(
        b["bk_tgt_mean"].abs() > 0.1,
        lag1 / b["bk_tgt_mean"].clip(0.1),
        1.0
    )
    b["bk_tgt_gap_pct"] = np.where(
        b["bk_tgt_mean"].abs() > 0.1,
        (b["bk_tgt_mean"] - lag1) / (b["bk_tgt_mean"].abs() + 1),
        0.0
    )

    b["tgt_delta2"] = b["bk_tgt_lag1"].fillna(0) - b["bk_tgt_lag2"].fillna(0)
    b["tgt_accel"]  = b["tgt_delta1"] - b["tgt_delta2"]
    b["tgt_roll3_mean"] = grp["bk_tgt_mean"].transform(
        lambda x: x.shift(1).rolling(3, min_periods=1).mean()
    )
    b["tgt_roll3_std"] = grp["bk_tgt_mean"].transform(
        lambda x: x.shift(1).rolling(3, min_periods=2).std().fillna(0)
    )

    prev_tgt_err = b["bk_tgt_lag1"].fillna(0) - b["bk_mean_lag2"].fillna(0)
    b["tgt_error_delta"] = b["bk_tgt_error"] - prev_tgt_err
    b["tgt_error_roll3"] = grp["bk_tgt_error"].transform(
        lambda x: x.shift(1).rolling(3, min_periods=1).mean()
    )

    speed    = b["bk_delta1"].fillna(0).abs().clip(lower=0.01)
    distance = (b["bk_tgt_mean"] - lag1).abs()
    b["eta_to_target"] = (distance / speed).clip(upper=50)

    # Rank / süreç ilerleme
    b["blk_rank"]     = grp.cumcount()
    b["blk_rank_rev"] = grp.cumcount(ascending=False)
    b["cum_dur"]      = grp["block_dur_s"].cumsum()
    b["log_cum_dur"]  = np.log1p(b["cum_dur"])
    b["dur_ratio"]    = b["block_dur_s"] / b["proc_total_dur"].clip(lower=1)

    # Rank'a bağlı feature’lar
    b["rank_x_delta"] = b["blk_rank"] * b["bk_delta1"].fillna(0)
    b["fill_rate"]    = b["bk_delta1"].fillna(0) / b["block_dur_s"].clip(lower=1)

    # Dinamik geçiş
    b["dyn_transition"] = (
        grp["is_dynamic"].diff().fillna(0).abs() > 0
    ).astype(int)

    # Kimlik + basit türevler
    b["cmd_machine"] = b["machineid"] * 100 + b["commandno"]
    b["was_dynamic"] = grp["is_dynamic"].shift(1).fillna(0).astype(int)
    b["bk_range"]    = b["bk_max"] - b["bk_min"]

    # Geçmiş satır sayısı
    hist_rows = sum(b[f"nrows_lag{l}"].fillna(0) for l in range(1, 6))
    b["log_hist_rows"] = np.log1p(hist_rows)

    # Numerikleri doldur
    num_c = b.select_dtypes(include=[np.float32, np.float64, np.int32, np.int64]).columns
    b[num_c] = b[num_c].fillna(0)
    return b

# ───────────────────── Veri hazırlama ─────────────────────

print("VERİ HAZIRLAMA")
train_raw = pd.read_csv(CFG["TRAIN_PATH"])
train_raw["timestamp"] = pd.to_datetime(
    train_raw["timestamp"], format="mixed", utc=True
).dt.tz_localize(None)
train_raw = add_proc_key(train_raw)

for c in VALVE_COLS:
    if c in train_raw.columns:
        train_raw[c] = train_raw[c].astype("float32")

train_clean = train_raw[train_raw["bk_level"] > 0].copy()
train_clean = train_clean.sort_values(["proc_key", "timestamp"]).reset_index(drop=True)

num_cols = train_clean.select_dtypes(include=[np.float32, np.float64]).columns.tolist()
id_cols  = ["proc_key", "timestamp", "machineid", "batchkey", "commandno"]

agg_d = {c: "mean" for c in num_cols if c not in id_cols}
if agg_d:
    train_clean = (
        train_clean
        .groupby(["proc_key", "timestamp"], sort=False)
        .agg({
            **{c: "first" for c in id_cols
               if c in train_clean.columns and c not in ["proc_key", "timestamp"]},
            **agg_d
        })
        .reset_index()
    )
    train_clean = train_clean.sort_values(["proc_key", "timestamp"]).reset_index(drop=True)

train_clean = cast_float32(train_clean)
print(f"Temiz: {train_clean.shape} | Proses: {train_clean['proc_key'].nunique():,}")

blk_raw = detect_and_aggregate_blocks(train_clean)
print(f"Blok: {len(blk_raw):,}")

cmd_stats_df  = build_cmd_stats(blk_raw)
proc_prior_df = build_proc_prior(blk_raw)
blk           = build_features(blk_raw, cmd_stats_df, proc_prior_df)

# ───────────────────── Warmup lookup (ilk blok tahmini) ─────────────────────

proc_first_bk = (
    blk_raw.sort_values(["proc_key", "block_id"])
    .groupby(["machineid", "commandno"])
    .apply(lambda g: g.groupby("proc_key")["bk_mean"].first().mean())
    .reset_index(name="bk_first_mean")
)

cmd_first_lookup = {
    (int(r["machineid"]), int(r["commandno"])): float(r["bk_first_mean"])
    for _, r in proc_first_bk.iterrows()
}

global_fallback = float(blk_raw["bk_mean"].mean())

def make_warmup(use_true: bool = False):
    def warmup_fn(proc_df: pd.DataFrame):
        if use_true:
            # Leaky simülasyonda gerçek ilk blok değeri
            return [float(proc_df.iloc[0]["bk_mean"])]
        # Test / non‑leaky durumda lookup
        mid = int(proc_df["machineid"].iloc[0])
        cmd = int(proc_df["commandno"].iloc[0])
        wu  = cmd_first_lookup.get((mid, cmd), global_fallback)
        return [float(np.clip(wu, 0, 100))]
    return warmup_fn

# ───────────────────── Tip normalize ─────────────────────

def normalize_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    for c in df.columns:
        dt = df[c].dtype
        if pd.api.types.is_float_dtype(dt):
            df[c] = df[c].astype("float32")
        elif pd.api.types.is_integer_dtype(dt):
            if dt == "int64":
                df[c] = df[c].astype("int32")
    return df

blk = normalize_dtypes(blk)


# Feature listeleri
FEAT_INIT = [c for c in BLK_FEATURES if c in blk.columns]
CAT_INIT  = [c for c in BLK_CAT if c in FEAT_INIT]

FEAT = FEAT_INIT
CAT  = CAT_INIT

proc_fold = (blk.groupby("proc_key")["t_start"].min()
             .reset_index().sort_values("t_start").reset_index(drop=True))
proc_fold["fold"] = np.arange(len(proc_fold)) % CFG["N_FOLDS"]
blk = blk.merge(proc_fold[["proc_key", "fold"]], on="proc_key", how="left")

# ───────────────────── 5-fold CV + stacking ─────────────────────

MODEL_KEYS = ["lgb1", "lgb2", "xgb1", "xgb2", "cat1", "cat2", "lr1", "lr2"]
n      = len(blk)
oof    = {k: np.zeros(n, dtype=np.float32) for k in MODEL_KEYS}
models = {k: [] for k in MODEL_KEYS}

print("\n5-FOLD CV")
for fold in range(CFG["N_FOLDS"]):
    tr = blk["fold"] != fold
    va = blk["fold"] == fold

    X_tr, y_tr = blk.loc[tr, FEAT].fillna(0), blk.loc[tr, CFG["TARGET"]]
    X_va, y_va = blk.loc[va, FEAT].fillna(0), blk.loc[va, CFG["TARGET"]]
    w_va       = blk.loc[va, "n_rows"].values.astype(float)
    cat_idx    = [FEAT.index(c) for c in CAT if c in FEAT]

    print(f"\n── Fold {fold} ──")

    ds_tr = lgb.Dataset(X_tr, y_tr, categorical_feature=CAT, free_raw_data=False)
    ds_va = lgb.Dataset(X_va, y_va, free_raw_data=False)
    for name, params in [("lgb1", LGB1), ("lgb2", LGB2)]:
        m = lgb.train(
            params, ds_tr, num_boost_round=CFG["MAX_ROUNDS"], valid_sets=[ds_va],
            callbacks=[
                lgb.early_stopping(CFG["EARLY_STOP"], verbose=False),
                lgb.log_evaluation(9999)
            ]
        )
        oof[name][va] = m.predict(X_va)
        models[name].append(m)
        print(f"  {name}: wMAE={weighted_mae(y_va, oof[name][va], w_va):.4f}  iter={m.best_iteration}")

    dm_tr = xgb.DMatrix(X_tr, label=y_tr)
    dm_va = xgb.DMatrix(X_va, label=y_va)
    for name, params in [("xgb1", XGB1), ("xgb2", XGB2)]:
        m = xgb.train(
            params, dm_tr, num_boost_round=CFG["MAX_ROUNDS"],
            evals=[(dm_va, "val")], early_stopping_rounds=CFG["EARLY_STOP"],
            verbose_eval=False
        )
        oof[name][va] = m.predict(dm_va, iteration_range=(0, m.best_iteration + 1))
        models[name].append(m)
        print(f"  {name}: wMAE={weighted_mae(y_va, oof[name][va], w_va):.4f}  iter={m.best_iteration}")

    for name, params in [("cat1", CAT1), ("cat2", CAT2)]:
        m = CatBoostRegressor(**params)
        m.fit(X_tr, y_tr, eval_set=(X_va, y_va), cat_features=cat_idx,
              use_best_model=True, verbose=False)
        oof[name][va] = m.predict(X_va)
        models[name].append(m)
        print(f"  {name}: wMAE={weighted_mae(y_va, oof[name][va], w_va):.4f}")

    for name, alpha in [("lr1", LR1_ALPHA), ("lr2", LR2_ALPHA)]:
        m = Ridge(alpha=alpha)
        m.fit(X_tr, y_tr)
        oof[name][va] = m.predict(X_va)
        models[name].append(m)
        print(f"  {name}: wMAE={weighted_mae(y_va, oof[name][va], w_va):.4f}")

y_true     = blk[CFG["TARGET"]].values
w_all      = blk["n_rows"].values.astype(float)
w_all_norm = w_all / w_all.mean()
oof_stack  = np.column_stack([oof[k] for k in MODEL_KEYS])
meta       = Ridge(alpha=1.0)
meta.fit(oof_stack, y_true, sample_weight=w_all_norm)
oof_final  = meta.predict(oof_stack)

print("\nSTACK SONUÇLARI")
for k in MODEL_KEYS:
    print(f"  {k}: wMAE={weighted_mae(y_true, oof[k], w_all):.4f}")
print(f"  STACK: MAE={mean_absolute_error(y_true, oof_final):.4f}  "
      f"wMAE={weighted_mae(y_true, oof_final, w_all):.4f}")
print("  Coef:", " ".join([f"{k}={v:.3f}" for k, v in zip(MODEL_KEYS, meta.coef_)]))
for cmd in [19, 20, 21, 22]:
    m_ = blk["commandno"] == cmd
    if m_.sum() > 0:
        print(f"  Komut {cmd}: wMAE={weighted_mae(y_true[m_], oof_final[m_], w_all[m_]):.4f}  "
              f"n_blok={m_.sum()}  n_satir={int(w_all[m_].sum())}")

# ───────────────────── Lookup & prediction helper ─────────────────────

cmd_stats_lookup = {
    (int(r["machineid"]), int(r["commandno"])): {
        "cmd_bk_mean_global"  : r["cmd_bk_mean_global"],
        "cmd_bk_std_global"   : r["cmd_bk_std_global"],
        "cmd_gap_mean_global" : r["cmd_gap_mean_global"],
        "cmd_bk_med_global"   : r["cmd_bk_med_global"],
        "cmd_tgt_error_mean"  : r["cmd_tgt_error_mean"],
        "cmd_tgt_error_std"   : r["cmd_tgt_error_std"],
    } for _, r in cmd_stats_df.iterrows()
}
prev_proc_lookup = dict(zip(proc_prior_df["proc_key"], proc_prior_df["prev_proc_bk_mean"]))

def _predict_single(feat_dict: dict) -> float:
    X_np = np.array([feat_dict.get(f, 0.0) for f in FEAT],
                    dtype=np.float32).reshape(1, -1)
    X_df_cat = None
    preds_raw = []
    for k, mlist in models.items():
        if k.startswith("lgb"):
            preds_raw.append(float(np.mean([m.predict(X_np)[0] for m in mlist])))
        elif k.startswith("xgb"):
            dm = xgb.DMatrix(X_np, feature_names=FEAT)
            preds_raw.append(float(np.mean([
                m.predict(dm, iteration_range=(0, m.best_iteration + 1))[0]
                for m in mlist
            ])))
        elif k.startswith("cat"):
            if X_df_cat is None:
                X_df_cat = pd.DataFrame(X_np, columns=FEAT)
                for c in CAT:
                    if c in X_df_cat.columns:
                        X_df_cat[c] = X_df_cat[c].astype(int)
            preds_raw.append(float(np.mean([m.predict(X_df_cat)[0] for m in mlist])))
        else:
            preds_raw.append(float(np.mean([m.predict(X_np)[0] for m in mlist])))
    return float(np.clip(meta.predict([preds_raw])[0], 0, 100))

# ───────────────────── Recursive block predict ─────────────────────

def recursive_block_predict(proc_blk_df: pd.DataFrame,
                            warmup_values=None) -> np.ndarray:
    blks  = proc_blk_df.reset_index(drop=True)
    n     = len(blks)
    preds = np.zeros(n, dtype=np.float32)
    hist  = {"bk": [], "std": [], "gap": [], "dyn": [], "nrows": [], "dur": [], "tgt": []}

    cmd = int(blks["commandno"].iloc[0])
    mid = int(blks["machineid"].iloc[0])
    cs  = cmd_stats_lookup.get((mid, cmd), {})

    proc_n_blocks  = n
    proc_n_dynamic = int(blks["is_dynamic"].sum())
    proc_total_dur = float(blks["block_dur_s"].sum())
    proc_total_gap = float(blks["gap_before_s"].sum())
    proc_tgt_mean  = float(blks["bk_tgt_mean"].mean())
    cum_dur        = 0.0

    blks_np = {c: blks[c].values for c in blks.columns}

    def hget(arr, lag, fallback=0.0):
        return arr[-lag] if len(arr) >= lag else (arr[-1] if arr else fallback)

    for i in range(n):
        h = hist["bk"]

        if warmup_values is not None and i < len(warmup_values):
            v = float(warmup_values[i])
            preds[i] = v
            hist["bk"].append(v)
            hist["std"].append(float(blks_np["bk_std"][i]))
            hist["gap"].append(float(blks_np["gap_before_s"][i]))
            hist["dyn"].append(int(blks_np["is_dynamic"][i]))
            hist["nrows"].append(float(blks_np["n_rows"][i]))
            hist["dur"].append(float(blks_np["block_dur_s"][i]))
            hist["tgt"].append(float(blks_np["bk_tgt_mean"][i]))
            cum_dur += float(blks_np["block_dur_s"][i])
            continue

        lag1 = hget(h, 1)
        tgt  = float(blks_np["bk_tgt_mean"][i])
        gap  = float(blks_np["gap_before_s"][i])
        cur_dur = float(blks_np["block_dur_s"][i])
        cum_dur += cur_dur

        fd = {}

        for lag in range(1, 6):
            fd[f"bk_mean_lag{lag}"] = hget(h, lag)
            fd[f"bk_last_lag{lag}"] = hget(h, lag)
            fd[f"bk_std_lag{lag}"]  = hget(hist["std"], lag)
            fd[f"gap_lag{lag}"]     = hget(hist["gap"], lag)
            fd[f"dyn_lag{lag}"]     = hget(hist["dyn"], lag)
            fd[f"dur_lag{lag}"]     = hget(hist["dur"], lag)
            fd[f"nrows_lag{lag}"]   = hget(hist["nrows"], lag)

        fd["bk_roll3_mean"]  = float(np.mean(h[-3:])) if len(h) >= 3 else lag1
        fd["bk_roll5_mean"]  = float(np.mean(h[-5:])) if len(h) >= 5 else lag1
        fd["bk_roll5_std"]   = float(np.std(h[-5:])) if len(h) >= 5 else 0.0
        fd["gap_roll3_mean"] = float(np.mean(hist["gap"][-3:])) if len(hist["gap"]) >= 3 else 0.0

        prev_tgt   = hget(hist["tgt"], 1, tgt)
        prev_tgt2  = hget(hist["tgt"], 2, tgt)
        tgt_d1     = tgt - prev_tgt
        tgt_d2     = prev_tgt - prev_tgt2

        fd["bk_tgt_mean"]  = tgt
        fd["bk_tgt_last"]  = float(blks_np["bk_tgt_last"][i])
        fd["bk_tgt_first"] = float(blks_np["bk_tgt_first"][i])
        fd["bk_tgt_lag1"]  = prev_tgt
        fd["bk_tgt_lag2"]  = prev_tgt2

        fd["bk_tgt_error"] = tgt - lag1
        fd["tgt_delta1"]   = tgt_d1
        fd["tgt_delta2"]   = tgt_d2
        fd["tgt_accel"]    = tgt_d1 - tgt_d2
        fd["tgt_roll3_mean"] = float(np.mean(hist["tgt"][-3:])) if len(hist["tgt"]) >= 3 else tgt
        fd["tgt_roll3_std"]  = float(np.std(hist["tgt"][-3:])) if len(hist["tgt"]) >= 3 else 0.0
        prev_tgt_err          = prev_tgt - hget(h, 2)
        fd["tgt_error_delta"] = (tgt - lag1) - prev_tgt_err
        fd["tgt_error_roll3"] = float(
            np.mean([hget(h, l, tgt) - tgt for l in range(1, 4)])
        ) if h else 0.0
        fd["bk_to_tgt_ratio"] = lag1 / max(abs(tgt), 0.1) if abs(tgt) > 0.1 else 1.0
        fd["bk_tgt_gap_pct"]  = ((tgt - lag1) / (abs(tgt) + 1)) if abs(tgt) > 0.1 else 0.0
        speed = max(abs(lag1 - hget(h, 2, lag1)), 0.01)
        fd["eta_to_target"] = min(abs(tgt - lag1) / speed, 50)

        fd["log_gap"]       = np.log1p(gap)
        fd["log_gap1"]      = np.log1p(hget(hist["gap"], 1))
        fd["log_gap3"]      = np.log1p(hget(hist["gap"], 3))
        fd["gap_x_bk_lag1"] = gap * lag1 / 3600
        fd["gap_x_tgt"]     = gap * tgt / 3600
        fd["cmd_gap_ratio"] = gap / (cs.get("cmd_gap_mean_global", 100.0) + 1)
        fd["log_total_gap"] = np.log1p(proc_total_gap)
        fd["gap_before_s"]  = gap
        fd["gap_x_dynamic"] = gap * int(blks_np["is_dynamic"][i])
        fd["gap_x_delta"]   = gap * abs(lag1 - hget(h, 2, lag1))

        fd["cmd_bk_mean_global"]  = cs.get("cmd_bk_mean_global",  lag1)
        fd["cmd_bk_std_global"]   = cs.get("cmd_bk_std_global",   1.0)
        fd["cmd_gap_mean_global"] = cs.get("cmd_gap_mean_global", 100.0)
        fd["cmd_bk_med_global"]   = cs.get("cmd_bk_med_global",   lag1)
        fd["cmd_tgt_error_mean"]  = cs.get("cmd_tgt_error_mean",  0.0)
        fd["cmd_tgt_error_std"]   = cs.get("cmd_tgt_error_std",   1.0)

        fd["proc_n_blocks"]  = proc_n_blocks
        fd["proc_n_dynamic"] = proc_n_dynamic
        fd["proc_total_dur"] = proc_total_dur
        fd["proc_total_gap"] = proc_total_gap
        fd["proc_tgt_mean"]  = proc_tgt_mean
        fd["proc_bk_start"]  = h[0] if h else 0.0
        fd["proc_bk_range"]  = (max(h) - min(h)) if len(h) >= 2 else 0.0

        fd["blk_rank"]     = i
        fd["blk_rank_rev"] = n - i - 1
        fd["log_cum_dur"]  = np.log1p(cum_dur)

        fd["machineid"]   = mid
        fd["commandno"]   = cmd
        fd["cmd_machine"] = mid * 100 + cmd

        fd["is_dynamic"]      = int(blks_np["is_dynamic"][i])
        fd["was_dynamic"]     = hget(hist["dyn"], 1)
        fd["n_rows"]          = float(blks_np["n_rows"][i])
        fd["block_dur_s"]     = cur_dur
        fd["bk_range"]        = float(blks_np["bk_max"][i]) - float(blks_np["bk_min"][i])
        fd["bk_draining"]     = int(lag1 - hget(h, 2, lag1) < -1)
        fd["log_hist_rows"]   = np.log1p(sum(hget(hist["nrows"], l) for l in range(1, 6)))

        fd["kk_mean"]  = float(blks_np["kk_mean"][i])
        fd["kk_last"]  = float(blks_np["kk_last"][i])
        fd["ak_mean"]  = float(blks_np["ak_mean"][i])
        fd["kk_robot_mean"] = float(blks_np["kk_robot_mean"][i])
        fd["bk_robot_mean"] = float(blks_np["bk_robot_mean"][i])
        fd["prev_proc_bk_mean"] = float(blks_np["prev_proc_bk_mean"][i]) if "prev_proc_bk_mean" in blks_np else 0.0
        fd["prev_proc_bk_last"] = float(blks_np["prev_proc_bk_last"][i]) if "prev_proc_bk_last" in blks_np else 0.0

        fd["bk_irtibat_ratio"]  = float(blks_np["bk_irtibat_ratio"][i])
        fd["fast_dosage_ratio"] = float(blks_np["fast_dosage_ratio"][i])
        fd["slow_dosage_ratio"] = float(blks_np["slow_dosage_ratio"][i])
        fd["bk_valve_ratio"]    = float(blks_np["bk_valve_ratio"][i])

        fd["kk_tgt_mean"] = float(blks_np["kk_tgt_mean"][i]) if "kk_tgt_mean" in blks_np else 0.0
        fd["kk_dosage_v"] = float(blks_np["kk_dosage_v"][i]) if "kk_dosage_v" in blks_np else 0.0
        fd["kk_irtibat_v"] = float(blks_np["kk_irtibat_v"][i]) if "kk_irtibat_v" in blks_np else 0.0
        fd["kk_bk_common"] = float(blks_np["kk_bk_common"][i]) if "kk_bk_common" in blks_np else 0.0

        fd["prgno"]         = int(blks_np["prgno"][i]) if "prgno" in blks_np else 0
        fd["fabric_weight"] = float(blks_np["fabric_weight"][i]) if "fabric_weight" in blks_np else 0.0
        fd["dosage_curve"]  = int(blks_np["dosage_curve"][i]) if "dosage_curve" in blks_np else 0
        fd["cmd_rep"]       = int(blks_np["cmd_rep"][i]) if "cmd_rep" in blks_np else 0

        pred = _predict_single(fd)
        preds[i] = pred

        hist["bk"].append(pred)
        hist["std"].append(float(blks_np["bk_std"][i]))
        hist["gap"].append(gap)
        hist["dyn"].append(int(blks_np["is_dynamic"][i]))
        hist["nrows"].append(float(blks_np["n_rows"][i]))
        hist["dur"].append(cur_dur)
        hist["tgt"].append(tgt)

    return preds

def run_recursive(blk_df: pd.DataFrame, warmup_fn=None, desc="Inference") -> np.ndarray:
    preds     = np.zeros(len(blk_df), dtype=np.float32)
    proc_keys = list(blk_df.groupby("proc_key").groups.keys())
    for pk in tqdm(proc_keys, desc=desc, ncols=80):
        proc_df = blk_df[blk_df["proc_key"] == pk]
        idx     = blk_df.index.get_indexer(proc_df.index)
        wu      = warmup_fn(proc_df) if warmup_fn else None
        preds[idx] = recursive_block_predict(proc_df, warmup_values=wu)
    return preds

# ───────────────────── Submission ─────────────────────

print("\nSUBMISSION")
test_raw = pd.read_csv(CFG["TEST_PATH"])
test_raw["timestamp"] = pd.to_datetime(
    test_raw["timestamp"], format="mixed", utc=True
).dt.tz_localize(None)
test_raw = add_proc_key(test_raw)

for c in VALVE_COLS:
    if c in test_raw.columns:
        test_raw[c] = test_raw[c].astype("float32")

test_raw = cast_float32(test_raw)
test_raw = test_raw.sort_values(["proc_key", "timestamp"]).reset_index(drop=True)

test_blk_raw = detect_and_aggregate_blocks(test_raw)
test_prior   = build_proc_prior(test_blk_raw)
prev_proc_lookup.update(dict(zip(test_prior["proc_key"], test_prior["prev_proc_bk_mean"])))

test_blk = build_features(test_blk_raw, cmd_stats_df, test_prior)
test_blk = normalize_dtypes(test_blk)

def get_test_rows_with_blocks(raw_df: pd.DataFrame) -> pd.DataFrame:
    df = raw_df.copy()
    df = fill_missing_sensor_cols(df)
    df["dt"] = df.groupby("proc_key")["timestamp"].diff().dt.total_seconds().fillna(0)

    def rs_cmd(grp):
        thr = CFG["CMD_DYN_THR"].get(int(grp["commandno"].iloc[0]), 1.0)
        return (grp["bk_level"].rolling(5, min_periods=2)
                .std().fillna(0) > thr).astype(int)

    df["is_dynamic"] = df.groupby("proc_key", group_keys=False).apply(rs_cmd)
    df["time_break"] = df.apply(
        lambda r: int(r["dt"] > CFG["CMD_GAP_THR"].get(int(r["commandno"]), 30)),
        axis=1
    )
    fi = df.groupby("proc_key").head(1).index
    df.loc[fi, "time_break"] = 1
    df["region_change"] = df.groupby("proc_key")["is_dynamic"].diff().fillna(0).abs().astype(int)
    df["new_block"]     = ((df["time_break"] == 1) | (df["region_change"] == 1)).astype(int)
    df.loc[fi, "new_block"] = 1
    df["block_id"]  = df.groupby("proc_key")["new_block"].cumsum()
    df["block_key"] = df["proc_key"] + "_b" + df["block_id"].astype(str)
    return df

test_rows  = get_test_rows_with_blocks(test_raw)
test_preds = run_recursive(test_blk, warmup_fn=make_warmup(use_true=False), desc="Test Submission")
pred_map   = dict(zip(
    test_blk["proc_key"] + "_b" + test_blk["block_id"].astype(str),
    test_preds
))

test_rows["bk_level_pred"] = test_rows["block_key"].map(pred_map).fillna(0)
sub = (test_rows[[CFG["ROW_ID_COL"], "bk_level_pred"]]
       .rename(columns={CFG["ROW_ID_COL"]: "Id", "bk_level_pred": "bk_level"})
       .sort_values("Id").reset_index(drop=True))
sub.to_csv("submission.csv", index=False)
print(f"Kaydedildi: {sub.shape}  min={sub['bk_level'].min():.2f}  "
      f"max={sub['bk_level'].max():.2f}  mean={sub['bk_level'].mean():.2f}")

VERİ HAZIRLAMA
Temiz: (2110822, 17) | Proses: 1,878
Blok: 16,032

5-FOLD CV

── Fold 0 ──
[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


[9999]	valid_0's l1: 1.03256
  lgb1: wMAE=0.8201  iter=15000
  lgb2: wMAE=0.9329  iter=3610
  xgb1: wMAE=0.7726  iter=10015
  xgb2: wMAE=0.7259  iter=4682


Default metric period is 5 because MAE is/are not implemented for GPU


  cat1: wMAE=0.6627


...

  cat1: wMAE=0.6558


Default metric period is 5 because MAE is/are not implemented for GPU


  cat2: wMAE=0.4300
  lr1: wMAE=6.4655
  lr2: wMAE=6.4668

STACK SONUÇLARI
  lgb1: wMAE=0.8222
  lgb2: wMAE=0.8363
  xgb1: wMAE=0.8045
  xgb2: wMAE=0.7466
  cat1: wMAE=0.6168
  cat2: wMAE=0.4238
  lr1: wMAE=4.7930
  lr2: wMAE=4.7906
  STACK: MAE=0.5876  wMAE=0.3965
  Coef: lgb1=0.064 lgb2=0.035 xgb1=0.045 xgb2=0.147 cat1=0.130 cat2=0.588 lr1=0.967 lr2=-0.973
  Komut 19: wMAE=0.4812  n_blok=5164  n_satir=291029
  Komut 20: wMAE=1.4772  n_blok=276  n_satir=3366
  Komut 21: wMAE=0.4099  n_blok=6666  n_satir=179076
  Komut 22: wMAE=0.3777  n_blok=3926  n_satir=1637351

SUBMISSION


Test Submission: 100%|████████████████████████| 595/595 [14:01<00:00,  1.41s/it]


Kaydedildi: (635133, 2)  min=9.47  max=68.22  mean=35.52
